## Run GraphPCA (Python biopy311)

In [1]:
import GraphPCA as sg
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import squidpy as sq
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances as pair
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
ground_truth = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/metadata.tsv', delimiter='\t', index_col=0)
adata.obs['ground_truth'] = ground_truth['ground_truth']
adata.var_names_make_unique()

/tmp/ipykernel_50532/2513864922.py:1: FutureWarning: Use `squidpy.read.visium` instead.
  adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/vanhoan310/miniconda3/envs/biopy311/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
# https://github.com/YANG-ERA/GraphPCA-Turbo
true_labels = pd.factorize(adata.obs['ground_truth'])[0]
adata.obs['x'] = adata.obsm['spatial'][:, 0]
adata.obs['y'] = adata.obsm['spatial'][:, 1]
sc.pp.filter_genes(adata, min_cells=1)
sc.experimental.pp.normalize_pearson_residuals(adata)
sc.pp.scale(adata)
x_array=adata.obs["x"].tolist()
y_array=adata.obs["y"].tolist()
location=np.array([x_array, y_array]).T.astype(np.float32)

In [4]:
print('Run GPCA')
Z, W = sg.Run_GPCA(adata,location=location,n_components=30,_lambda=0.5,n_neighbors=6,mode="iterative",random_seed=0)
print('Run clustering')
adata.obsm["GraphPCA"] = Z
estimator = KMeans(n_clusters=len(np.unique(true_labels)))
res = estimator.fit(Z[:,:])
lable_pred=res.labels_
adata.obs["GPCA_pred"]= lable_pred
adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')
refined_pred=sg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["GPCA_pred"].tolist(), dis= pair(location),
                       shape="generic",neighbor_num=7)
adata.obs["GPCA_pred"]= refined_pred
adata.obs["GPCA_pred"] = adata.obs["GPCA_pred"].astype('category')

Run GPCA


Reached maximum iterations (5) without convergence.


Run clustering


In [5]:
NMI = normalized_mutual_info_score(adata.obs['GPCA_pred'], adata.obs['ground_truth'])
print(NMI)

0.6523722682676734


In [6]:
adata.write_h5ad(
    '/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/Section_1_GraphPCA.h5ad'
)